# Phase 2 — Task 1: Trend Prediction Probing (All 6 Baseline Models)

**Protocol (from `phase2-downstream-probes.md`):**
* Evaluates what each frozen encoder's latent representation $z \in \mathbb{R}^{256}$ already knows about future price movements.
* Mid-price: $\text{mid}(t) = \frac{\text{BidPrice1}(t) + \text{AskPrice1}(t)}{2}$
* Horizon: $k = 5$ ticks (15s at 3s sampling).
* Raw signal: $l(t) = \overline{\text{mid}}_{\text{future}} - \overline{\text{mid}}_{\text{past}}$
* Threshold $\theta$: 33rd / 67th percentile computed on the **TRAINING SPLIT ONLY** (no data leakage).
* Probe Head: Single linear layer `nn.Linear(256, 3)` (Adam lr=1e-3, 50 epochs).
* Primary Metric: **Macro-F1** (along with Accuracy, Precision, Recall).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
!pip install -q lightning pandas numpy torch scikit-learn
print("✓ Environment and Google Drive ready.")


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
from downstream_common import (
    MODEL_REGISTRY, STOCKS, LATENT_DIM, set_seed,
    precompute_and_cache_latents, train_trend_head_probe
)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
os.makedirs("downstream_results", exist_ok=True)
os.makedirs("latents", exist_ok=True)


In [ ]:
# 1. Ensure Latents are Cached for all 30 Experiments
for model_name in MODEL_REGISTRY.keys():
    for stock in STOCKS:
        train_path = f"latents/{model_name}/{stock}/train_latents.npy"
        if not os.path.exists(train_path):
            print(f"Precomputing latents for {model_name} / {stock} ...")
            precompute_and_cache_latents(model_name, stock, out_dir="latents", device=device)
        else:
            print(f"✓ Found cached latents: {model_name}/{stock}")
print("\nAll 30 latent representations ready.")


In [ ]:
# Run Trend Prediction Across All 30 Experiments
out_csv = "downstream_results/trend_prediction_results.csv"
if os.path.exists(out_csv):
    df_results = pd.read_csv(out_csv)
    results = df_results.to_dict('records')
    done_pairs = set((r['model'], r['stock']) for r in results)
else:
    results = []
    done_pairs = set()

print("=" * 80)
print("  TASK 1: TREND PREDICTION PROBING (Macro-F1 & Accuracy)")
print("=" * 80)

for model_name in MODEL_REGISTRY.keys():
    print(f"\nEvaluating Model: {model_name}")
    for stock in STOCKS:
        if (model_name, stock) in done_pairs:
            print(f"  {model_name:<12} / {stock}: (already completed, skipping)")
            continue
            
        train_z = np.load(f"latents/{model_name}/{stock}/train_latents.npy")
        train_y = np.load(f"latents/{model_name}/{stock}/train_labels.npy")
        val_z   = np.load(f"latents/{model_name}/{stock}/val_latents.npy")
        val_y   = np.load(f"latents/{model_name}/{stock}/val_labels.npy")
        test_z  = np.load(f"latents/{model_name}/{stock}/test_latents.npy")
        test_y  = np.load(f"latents/{model_name}/{stock}/test_labels.npy")
        thetas  = np.load(f"latents/{model_name}/{stock}/thetas.npy") if os.path.exists(f"latents/{model_name}/{stock}/thetas.npy") else np.load(f"latents/{model_name}/{stock}/theta.npy")
        theta_down, theta_up = float(thetas[0]), float(thetas[1]) if len(thetas) > 1 else float(thetas[0])
        
        t0 = time.time()
        metrics = train_trend_head_probe(
            train_z, train_y, val_z, val_y, test_z, test_y,
            epochs=50, lr=1e-3, batch_size=256, device=device
        )
        elapsed = time.time() - t0
        
        print(f"  {model_name:<12} / {stock}: Macro-F1 = {metrics['macro_f1']:.4f}, Acc = {metrics['accuracy']:.4f} (θ_down={theta_down:.5f}, θ_up={theta_up:.5f}, {elapsed:.1f}s)")
        
        row = {
            'model': model_name,
            'stock': stock,
            'macro_f1': metrics['macro_f1'],
            'accuracy': metrics['accuracy'],
            'prec_down': metrics['precision_down'],
            'rec_down': metrics['recall_down'],
            'prec_stable': metrics['precision_stable'],
            'rec_stable': metrics['recall_stable'],
            'prec_up': metrics['precision_up'],
            'rec_up': metrics['recall_up'],
            'theta_down': theta_down,
            'theta_up': theta_up
        }
        results.append(row)
        done_pairs.add((model_name, stock))
        pd.DataFrame(results).to_csv(out_csv, index=False)

df_results = pd.DataFrame(results)
print("\n✓ Saved: downstream_results/trend_prediction_results.csv")


In [ ]:
# 3. Formatted Benchmark Summary Table
pivot_f1 = df_results.pivot(index='model', columns='stock', values='macro_f1')
pivot_f1['Mean Macro-F1'] = pivot_f1.mean(axis=1)
pivot_f1 = pivot_f1.sort_values(by='Mean Macro-F1', ascending=False)

print("=" * 80)
print("  TASK 1 SUMMARY: TEST MACRO-F1 RANKING")
print("=" * 80)
display(pivot_f1.round(4))
